In [3]:
!pip install -q transformers datasets accelerate

In [6]:
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [8]:
from datasets import load_dataset

raw = load_dataset("fancyzhx/amazon_polarity", split="train[:5000]")

positive = [r["content"] for r in raw if r["label"] == 1][:500]
print(f"Collected {len(positive)} positive reviews")
print(positive[0][:200])

with open("reviews.txt", "w", encoding="utf-8") as f:
    for review in positive:
        f.write(review.strip().replace("\n", " ") + "\n")

README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

amazon_polarity/train-00002-of-00004.par(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

amazon_polarity/test-00000-of-00001.parq(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

Collected 500 positive reviews
This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I


In [9]:
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
print(f"Loaded GPT-2 with {model.num_parameters():,} parameters")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded GPT-2 with 124,439,808 parameters


In [11]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

from datasets import load_dataset
dataset = load_dataset("text", data_files={"train": "reviews.txt"})

train_dataset = dataset["train"].map(tokenize_function, batched=True)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [13]:
training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=1,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.818192
100,3.575317
150,3.407837


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=3.5459911205150463, metrics={'train_runtime': 3200.1593, 'train_samples_per_second': 0.469, 'train_steps_per_second': 0.059, 'total_flos': 96704589312000.0, 'train_loss': 3.5459911205150463, 'epoch': 3.0})

In [14]:
model.eval()

def generate_review(prompt, max_length=60, temperature=0.8):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    output = model.generate(
        input_ids,
        max_length=max_length,
        temperature=temperature,
        top_k=50,
        top_p=0.95,
        do_sample=True,
        num_return_sequences=3,
        pad_token_id=tokenizer.eos_token_id
    )
    return [tokenizer.decode(o, skip_special_tokens=True) for o in output]

for review in generate_review("This product is"):
    print("—", review)
    print()

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


— This product is great and the price is amazing! I have been using it for over a year and love it so much! It also works great with my iPod or iPad! I recommend this product for the iPod player and iMac. If you have two iPod's and need to play on the same

— This product is great for beginners and professionals alike. My daughter loves it and is looking forward to getting her hands on it for her first time. I highly recommend this product to anyone wanting a great tool for any project. It is a great product that is portable and easy to get on and off,

— This product is very good, however for those who don't need it the color changes pretty quickly. It is not sticky and doesn't get sticky on the skin. This product is much better than the others. I love it because it helps me avoid eye irritation. It doesn't get sticky when I



In [15]:
prompts = ["This product is", "I bought this for", "The quality of this item"]
for p in prompts:
    print(f"PROMPT: {p}")
    for r in generate_review(p, max_length=50):
        print("  ", r)
    print()

PROMPT: This product is
   This product is so great it's perfect for a hot summer day. My husband and I are already wearing it. We love it! We have a small child and love it! We can't wait to wear it! Thank you SO much! So
   This product is a great addition to your wardrobe. I wear it for a very short period of time and it keeps my hair in perfect shape. I wore it for two weeks and it is now back in place. It has also worn well in the
   This product is amazing! My daughter loves it and she does not have to wear it all day. It is great for a quick change of clothes to look more comfortable. I wear it around the house in the evenings and have to wear it while I

PROMPT: I bought this for
   I bought this for my daughter and she loves it. She loves it, has a lot of fun, and likes it when she watches movies. She has no idea what she is doing wrong. This is a great product that I would highly recommend to
   I bought this for my dog who is allergic to this stuff. He loves it and loves to 

In [16]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method4

def bleu_against_corpus(generated_text, reference_reviews):
    ref_tokens = [r.split() for r in reference_reviews]
    gen_tokens = generated_text.split()
    return sentence_bleu(ref_tokens, gen_tokens, smoothing_function=smoothie)

sample_gen = generate_review("This product is")[0]
score = bleu_against_corpus(sample_gen, positive[:50])
print(f"BLEU score: {score:.3f}")

BLEU score: 0.076
